In [ ]:
import os

import numpy as np
import pandas as pd
import yfinance as yf

# ------------------------------------------------------------------------------
# 1. DEFINE TARGET TICKER UNIVERSE WITH METADATA TAGS
# ------------------------------------------------------------------------------
# Mapping tickers to Sector/Category for downstream comparative analysis
TICKER_METADATA = {
    # --- 2008 Global Financial Crisis (Distressed & Anchors) ---
    "LEHMQ": {
        "Name": "Lehman Brothers",
        "Category": "Distressed_2008",
        "Sector": "Financials",
    },
    "BSC": {
        "Name": "Bear Stearns",
        "Category": "Distressed_2008",
        "Sector": "Financials",
    },
    "AIG": {
        "Name": "American International Group",
        "Category": "Distressed_2008",
        "Sector": "Financials",
    },
    "C": {"Name": "Citigroup", "Category": "Distressed_2008", "Sector": "Financials"},
    "JPM": {
        "Name": "JPMorgan Chase",
        "Category": "Anchor_Financial",
        "Sector": "Financials",
    },
    "BAC": {
        "Name": "Bank of America",
        "Category": "Anchor_Financial",
        "Sector": "Financials",
    },
    "GS": {
        "Name": "Goldman Sachs",
        "Category": "Anchor_Financial",
        "Sector": "Financials",
    },
    "MS": {
        "Name": "Morgan Stanley",
        "Category": "Anchor_Financial",
        "Sector": "Financials",
    },
    # --- 2020 COVID Crisis (Distressed Specialty Financials & Energy) ---
    "MFA": {
        "Name": "MFA Financial",
        "Category": "Distressed_2020",
        "Sector": "Real_Estate_Finance",
    },
    "IVR": {
        "Name": "Invesco Mortgage Capital",
        "Category": "Distressed_2020",
        "Sector": "Real_Estate_Finance",
    },
    "TWO": {
        "Name": "Two Harbors Investment",
        "Category": "Distressed_2020",
        "Sector": "Real_Estate_Finance",
    },
    "HTZ": {
        "Name": "Hertz Global Holdings",
        "Category": "Distressed_2020",
        "Sector": "Consumer_Services",
    },
    "CHK": {
        "Name": "Chesapeake Energy",
        "Category": "Distressed_2020",
        "Sector": "Energy",
    },
    # --- 2023 Regional Banking Crisis (Distressed & Regional Anchors) ---
    "SIVB": {
        "Name": "Silicon Valley Bank",
        "Category": "Distressed_2023",
        "Sector": "Financials",
    },
    "FRCB": {
        "Name": "First Republic Bank",
        "Category": "Distressed_2023",
        "Sector": "Financials",
    },
    "SBNY": {
        "Name": "Signature Bank",
        "Category": "Distressed_2023",
        "Sector": "Financials",
    },
    "PACW": {
        "Name": "PacWest Bancorp",
        "Category": "Distressed_2023",
        "Sector": "Financials",
    },
    "WAL": {
        "Name": "Western Alliance",
        "Category": "Regional_Bank",
        "Sector": "Financials",
    },
    "KEY": {"Name": "KeyCorp", "Category": "Regional_Bank", "Sector": "Financials"},
    "FITB": {
        "Name": "Fifth Third Bancorp",
        "Category": "Regional_Bank",
        "Sector": "Financials",
    },
    # --- Market Indices & Sector Benchmarks ---
    "^GSPC": {
        "Name": "S&P 500 Index",
        "Category": "Benchmark",
        "Sector": "Broad_Market",
    },
    "^VIX": {
        "Name": "CBOE Volatility Index",
        "Category": "Benchmark",
        "Sector": "Volatility",
    },
    "XLF": {
        "Name": "Financial Select Sector SPDR",
        "Category": "Benchmark",
        "Sector": "Financials",
    },
    "XLK": {
        "Name": "Technology Select Sector SPDR",
        "Category": "Control_Sector",
        "Sector": "Technology",
    },
    "XLE": {
        "Name": "Energy Select Sector SPDR",
        "Category": "Control_Sector",
        "Sector": "Energy",
    },
}

TICKERS = list(TICKER_METADATA.keys())
START_DATE = "2006-01-01"  # Fetching extra buffer data prior to GFC 2007
END_DATE = "2023-12-31"

print(
    f"Downloading historical data for {len(TICKERS)} tickers from {START_DATE} to {END_DATE}..."
)

# ------------------------------------------------------------------------------
# 2. BATCH DOWNLOAD HISTORICAL PRICES VIA YFINANCE
# ------------------------------------------------------------------------------
# Use yf.download for fast, vectorized batch retrieval
raw_data = yf.download(
    tickers=TICKERS,
    start=START_DATE,
    end=END_DATE,
    interval="1d",
    group_by="ticker",
    auto_adjust=False,
    threads=True,
)

# ------------------------------------------------------------------------------
# 3. TRANSFORM & CLEAN INTO TIDY LONG-FORMAT DATAFRAME
# ------------------------------------------------------------------------------
records = []
missing_tickers = []

for ticker in TICKERS:
    try:
        # Check if ticker returned data
        if ticker in raw_data.columns.levels[0]:
            df_ticker = raw_data[ticker].copy().dropna(how="all")

            if df_ticker.empty:
                missing_tickers.append(ticker)
                continue

            df_ticker.reset_index(inplace=True)
            df_ticker["Ticker"] = ticker
            df_ticker["Name"] = TICKER_METADATA[ticker]["Name"]
            df_ticker["Category"] = TICKER_METADATA[ticker]["Category"]
            df_ticker["Sector"] = TICKER_METADATA[ticker]["Sector"]

            # Calculate daily percent log returns on Adjusted Close
            df_ticker["Adj Close"] = df_ticker["Adj Close"].astype(float)
            df_ticker["Log_Return"] = np.log(
                df_ticker["Adj Close"] / df_ticker["Adj Close"].shift(1)
            )

            records.append(df_ticker)
        else:
            missing_tickers.append(ticker)
    except Exception as e:
        print(f"Error processing {ticker}: {e}")
        missing_tickers.append(ticker)

# Concatenate all ticker data into a single master long-format DataFrame
master_df = pd.concat(records, ignore_index=True)

# Format columns
master_df.rename(columns={"Date": "Date", "Adj Close": "Adj_Close"}, inplace=True)
master_df.sort_values(by=["Ticker", "Date"], inplace=True)

# ------------------------------------------------------------------------------
# 4. SUMMARY & VALIDATION
# ------------------------------------------------------------------------------
print("\n" + "=" * 60)
print(f"SUCCESSFULLY DOWNLOADED DATA FOR {len(master_df['Ticker'].unique())} TICKERS.")
if missing_tickers:
    print(
        f"WARNING: The following tickers returned no data or are delisted on Yahoo: {missing_tickers}"
    )
    print(
        "Note: For delisted entities missing from Yahoo, offline CSV patches will be merged."
    )
print("=" * 60)

# Display sample output
print("\nMaster Dataset Preview:")
print(master_df[["Date", "Ticker", "Sector", "Adj_Close", "Log_Return"]].head())

# ------------------------------------------------------------------------------
# 5. EXPORT TO CSV AND PARQUET
# ------------------------------------------------------------------------------
output_dir = "../data/raw"
os.makedirs(output_dir, exist_ok=True)

csv_path = os.path.join(output_dir, "market_data_raw.csv")
parquet_path = os.path.join(output_dir, "market_data_raw.parquet")

# Save as CSV
master_df.to_csv(csv_path, index=False)
print(
    f"\n[Saved] Raw CSV created at: {csv_path} ({os.path.getsize(csv_path) / 1e6:.2f} MB)"
)

# Save as Parquet (Requires 'pyarrow' or 'fastparquet')
try:
    master_df.to_parquet(parquet_path, index=False)
    print(
        f"[Saved] Raw Parquet created at: {parquet_path} ({os.path.getsize(parquet_path) / 1e6:.2f} MB)"
    )
except ImportError:
    print(
        "\nTip: Install 'pyarrow' using `pip install pyarrow` to save directly in Parquet format."
    )

$BSC: possibly delisted; no price data found  (1d 2006-01-01 -> 2023-12-31)
[                       0%                       ]

[****                   8%                       ]  2 of 25 completed$CHK: possibly delisted; no timezone found
[********************* 44%                       ]  11 of 25 completed$LEHMQ: possibly delisted; no price data found  (1d 2006-01-01 -> 2023-12-31)
[**********************52%                       ]  13 of 25 completed$SBNY: possibly delisted; no price data found  (1d 2006-01-01 -> 2023-12-31) (Yahoo error = "Data doesn't exist for startDate = 1136091600, endDate = 1703998800")
[**********************72%**********             ]  18 of 25 completed$PACW: possibly delisted; no timezone found
[**********************72%**********             ]  18 of 25 completed$SIVB: possibly delisted; no timezone found
[*********************100%***********************]  25 of 25 completed

6 Failed downloads:
['BSC', 'LEHMQ']: possibly delisted; no price data found  (1d 2006-01-01 -> 2023-12-31)
['CHK', 'PACW', 'SIVB']: possibly delisted; no timezone found
['SBNY']: possibly delisted; no price


SUCCESSFULLY DOWNLOADED DATA FOR 19 TICKERS.
Note: For delisted entities missing from Yahoo, offline CSV patches will be merged.

Master Dataset Preview:
Price       Date Ticker      Sector   Adj_Close  Log_Return
0     2006-01-03    AIG  Financials  829.507202         NaN
1     2006-01-04    AIG  Financials  830.698914    0.001436
2     2006-01-05    AIG  Financials  831.771118    0.001290
3     2006-01-06    AIG  Financials  835.345459    0.004288
4     2006-01-09    AIG  Financials  831.413818   -0.004718

[Saved] Raw CSV created at: ../data/raw/market_data_raw.csv (14.30 MB)
[Saved] Raw Parquet created at: ../data/raw/market_data_raw.parquet (3.18 MB)


In [ ]:
import os

import numpy as np
import pandas as pd

# 1. Load your master raw dataset
parquet_path = "../data/raw/market_data_raw.parquet"
master_df = pd.read_parquet(parquet_path)

# 2. Define metadata for the historical truncated failure tickers
patch_metadata = {
    "LEHMQ": {
        "Name": "Lehman Brothers",
        "Category": "Distressed_2008",
        "Sector": "Financials",
    },
    "BSC": {
        "Name": "Bear Stearns",
        "Category": "Distressed_2008",
        "Sector": "Financials",
    },
    "SIVB": {
        "Name": "Silicon Valley Bank",
        "Category": "Distressed_2023",
        "Sector": "Financials",
    },
    "SBNY": {
        "Name": "Signature Bank",
        "Category": "Distressed_2023",
        "Sector": "Financials",
    },
    "PACW": {
        "Name": "PacWest Bancorp",
        "Category": "Distressed_2023",
        "Sector": "Financials",
    },
    "CHK": {
        "Name": "Chesapeake Energy",
        "Category": "Distressed_2020",
        "Sector": "Energy",
    },
}

patch_dir = "../data/patches/"
patch_records = []

# 3. Read valid patch CSV files and compute log returns
for ticker, meta in patch_metadata.items():
    file_path = os.path.join(patch_dir, f"{ticker}.csv")
    if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
        df_p = pd.read_csv(file_path)
        df_p["Date"] = pd.to_datetime(df_p["Date"])
        df_p = df_p.sort_values("Date").drop_duplicates(subset=["Date"])

        df_p["Ticker"] = ticker
        df_p["Name"] = meta["Name"]
        df_p["Category"] = meta["Category"]
        df_p["Sector"] = meta["Sector"]

        # Calculate log returns up to truncation
        df_p["Log_Return"] = np.log(df_p["Adj_Close"] / df_p["Adj_Close"].shift(1))
        patch_records.append(df_p)
        print(
            f"[Patched] Loaded truncated history for {ticker} ({len(df_p)} trading days)."
        )

# 4. Merge patches into master dataset if any exist
if patch_records:
    patch_df = pd.concat(patch_records, ignore_index=True)
    patched_symbols = list(patch_metadata.keys())

    # Remove any existing partial entries and append clean truncated histories
    master_df = master_df[~master_df["Ticker"].isin(patched_symbols)]
    master_df = pd.concat([master_df, patch_df], ignore_index=True)
    master_df = master_df.sort_values(by=["Ticker", "Date"]).reset_index(drop=True)

    # Overwrite master raw dataset
    master_df.to_parquet(parquet_path, engine="pyarrow", index=False)
    print("\n[Success] Master raw dataset updated with truncated failure histories!")


[Feature Engine Complete] Processed dataset saved to: ../data/processed/market_data_features_delisted.parquet
Total Tickers in Final Dataset: 19
Total Distress Trigger Events (Label=1): 875


In [ ]:
###
###
###

### SCRATCH WORK THAT DIDNT WORK BELOW ###

###
###
###

In [ ]:
# NO LONGER NEEDED: The following code snippet is now redundant since the CSV and Parquet files are already saved in the previous steps. It was originally intended to read the CSV and save it as Parquet, but this is now handled directly in the export step above.

import os

import pandas as pd

# Define exact paths
csv_path = "./data/raw/market_data_raw.csv"
parquet_path = "./data/raw/market_data_raw.parquet"

# Fallback check if CSV is in ./data/ instead of ./data/raw/
if not os.path.exists(csv_path) and os.path.exists("./data/market_data_raw.csv"):
    csv_path = "./data/market_data_raw.csv"

print(f"Reading CSV from: {csv_path}")

# Load the CSV
df = pd.read_csv(csv_path)
df["Date"] = pd.to_datetime(df["Date"])

# Ensure target folder exists
os.makedirs("./data/raw", exist_ok=True)

# Write directly to ./data/raw/
df.to_parquet(parquet_path, engine="pyarrow", index=False)

# Verify
if os.path.exists(parquet_path):
    size_mb = os.path.getsize(parquet_path) / 1e6
    print(
        f"SUCCESS! Parquet file written to: {parquet_path} (Size: {size_mb:.2f}" " MB)"
    )
    print("\nContents of ./data/raw/:")
    print(os.listdir("./data/raw/"))
else:
    print("Failed to write Parquet file.")

In [1]:
import pandas as pd

# Define the complete list of 25 tickers you intended to ingest
intended_tickers = [
    "LEHMQ",
    "BSC",
    "AIG",
    "C",
    "JPM",
    "BAC",
    "GS",
    "MS",  # 2008 GFC
    "SIVB",
    "FRCB",
    "SBNY",
    "PACW",
    "WAL",
    "KEY",
    "FITB",  # 2023 Banking Crisis
    "MFA",
    "IVR",
    "TWO",
    "HTZ",
    "CHK",  # 2020 COVID / Others
    "^GSPC",
    "^VIX",
    "XLF",
    "XLK",
    "XLE",  # Benchmarks
]

# Load your current master parquet dataset
df_features = pd.read_parquet("../data/processed/market_data_features.parquet")
actual_tickers = df_features["Ticker"].unique().tolist()

# Find the missing ones
missing_tickers = [t for t in intended_tickers if t not in actual_tickers]

print(f"Total Intended: {len(intended_tickers)}")
print(f"Total Successfully Ingested: {len(actual_tickers)}")
print(f"Missing Tickers: {missing_tickers}")

Total Intended: 25
Total Successfully Ingested: 19
Missing Tickers: ['LEHMQ', 'BSC', 'SIVB', 'SBNY', 'PACW', 'CHK']


In [6]:
import os

import numpy as np
import pandas as pd

parquet_path = "../data/raw/market_data_raw.parquet"
master_df = pd.read_parquet(parquet_path)

patch_metadata = {
    "LEHMQ": {
        "Name": "Lehman Brothers",
        "Category": "Distressed_2008",
        "Sector": "Financials",
    },
    "BSC": {
        "Name": "Bear Stearns",
        "Category": "Distressed_2008",
        "Sector": "Financials",
    },
    "SIVB": {
        "Name": "Silicon Valley Bank",
        "Category": "Distressed_2023",
        "Sector": "Financials",
    },
    "SBNY": {
        "Name": "Signature Bank",
        "Category": "Distressed_2023",
        "Sector": "Financials",
    },
    "PACW": {
        "Name": "PacWest Bancorp",
        "Category": "Distressed_2023",
        "Sector": "Financials",
    },
    "CHK": {
        "Name": "Chesapeake Energy",
        "Category": "Distressed_2020",
        "Sector": "Energy",
    },
}

patch_dir = "../data/patches/"
patch_records = []

for ticker, meta in patch_metadata.items():
    csv_file_path = os.path.join(patch_dir, f"{ticker}.csv")

    if os.path.exists(csv_file_path):
        # Check if file size is greater than 0 bytes
        if os.path.getsize(csv_file_path) > 0:
            try:
                df_patch = pd.read_csv(csv_file_path)

                # Verify required columns exist
                if "Date" in df_patch.columns and "Adj_Close" in df_patch.columns:
                    df_patch["Date"] = pd.to_datetime(df_patch["Date"])
                    df_patch["Ticker"] = ticker
                    df_patch["Name"] = meta["Name"]
                    df_patch["Category"] = meta["Category"]
                    df_patch["Sector"] = meta["Sector"]
                    df_patch["Log_Return"] = np.log(
                        df_patch["Adj_Close"] / df_patch["Adj_Close"].shift(1)
                    )

                    patch_records.append(df_patch)
                    print(f"[Loaded] Successfully patched {ticker}")
                else:
                    print(
                        f"[Warning] {ticker}.csv is missing required columns ('Date', 'Adj_Close')."
                    )
            except Exception as e:
                print(f"[Error] Could not parse {ticker}.csv: {e}")
        else:
            print(f"[Skipped] {ticker}.csv is empty (0 bytes). Add valid data to it.")
    else:
        print(f"[Skipped] No patch file found for {ticker}")

# If valid patches were processed, merge them
if patch_records:
    patch_df = pd.concat(patch_records, ignore_index=True)
    patched_tickers = [p["Ticker"].iloc[0] for p in patch_records]

    master_df = master_df[~master_df["Ticker"].isin(patched_tickers)]
    master_df = pd.concat([master_df, patch_df], ignore_index=True)
    master_df.sort_values(by=["Ticker", "Date"], inplace=True)

    master_df.to_parquet(parquet_path, engine="pyarrow", index=False)
    print(
        f"\n[Success] Master Parquet updated! Total unique tickers: {master_df['Ticker'].nunique()}"
    )
else:
    print("\nNo valid patch files were ready to merge yet.")

[Skipped] LEHMQ.csv is empty (0 bytes). Add valid data to it.
[Skipped] BSC.csv is empty (0 bytes). Add valid data to it.
[Skipped] SIVB.csv is empty (0 bytes). Add valid data to it.
[Skipped] SBNY.csv is empty (0 bytes). Add valid data to it.
[Skipped] PACW.csv is empty (0 bytes). Add valid data to it.
[Skipped] CHK.csv is empty (0 bytes). Add valid data to it.

No valid patch files were ready to merge yet.


In [7]:
import os

import numpy as np
import pandas as pd
import yfinance as yf

# Define the exact target delisted/distressed tickers that failed previously
# Note: For dead/delisted equities, we use historical symbols or standard equivalents
delisted_targets = {
    "LEHMQ": {
        "Name": "Lehman Brothers",
        "Category": "Distressed_2008",
        "Sector": "Financials",
        "start": "2006-01-03",
        "end": "2008-09-15",
    },
    "BSC": {
        "Name": "Bear Stearns",
        "Category": "Distressed_2008",
        "Sector": "Financials",
        "start": "2006-01-03",
        "end": "2008-03-17",
    },
    "SIVB": {
        "Name": "Silicon Valley Bank",
        "Category": "Distressed_2023",
        "Sector": "Financials",
        "start": "2018-01-02",
        "end": "2023-03-10",
    },
    "SBNY": {
        "Name": "Signature Bank",
        "Category": "Distressed_2023",
        "Sector": "Financials",
        "start": "2018-01-02",
        "end": "2023-03-12",
    },
    "PACW": {
        "Name": "PacWest Bancorp",
        "Category": "Distressed_2023",
        "Sector": "Financials",
        "start": "2018-01-02",
        "end": "2023-12-29",
    },
    "CHK": {
        "Name": "Chesapeake Energy",
        "Category": "Distressed_2020",
        "Sector": "Energy",
        "start": "2018-01-02",
        "end": "2020-06-30",
    },
}

patch_dir = "../data/patches/"
os.makedirs(patch_dir, exist_ok=True)

downloaded_patches = []

for ticker, meta in delisted_targets.items():
    print(
        f"Fetching historical data for {ticker} ({meta['Name']}) from {meta['start']} to {meta['end']}..."
    )
    try:
        # Download historical data from Yahoo Finance up to their respective crash/delisting date
        df_t = yf.download(ticker, start=meta["start"], end=meta["end"], progress=False)

        if not df_t.empty:
            # Flatten multi-index columns if returned by newer yfinance versions
            if isinstance(df_t.columns, pd.MultiIndex):
                df_t.columns = df_t.columns.get_level_values(0)

            df_t = df_t.reset_index()

            # Ensure standard column formatting
            if "Adj Close" in df_t.columns:
                df_t.rename(
                    columns={
                        "Adj Close": "Adj_Close",
                        "Close": "Close",
                        "Open": "Open",
                        "High": "High",
                        "Low": "Low",
                        "Volume": "Volume",
                    },
                    inplace=True,
                )
            elif "Close" in df_t.columns:
                df_t["Adj_Close"] = df_t["Close"]

            df_t["Ticker"] = ticker
            df_t["Name"] = meta["Name"]
            df_t["Category"] = meta["Category"]
            df_t["Sector"] = meta["Sector"]

            # Save individual patch CSV into data/patches/
            csv_out = os.path.join(patch_dir, f"{ticker}.csv")
            df_t.to_csv(csv_out, index=False)
            print(f" -> Successfully saved {ticker}.csv ({len(df_t)} rows)")
            downloaded_patches.append(df_t)
        else:
            print(
                f" -> Warning: Yahoo returned empty feed for {ticker}. Will need manual backup source."
            )
    except Exception as e:
        print(f" -> Error downloading {ticker}: {e}")

print(
    f"\nCompleted fetching patches. Total patches ready: {len(downloaded_patches)} / {len(delisted_targets)}"
)

Fetching historical data for LEHMQ (Lehman Brothers) from 2006-01-03 to 2008-09-15...


$LEHMQ: possibly delisted; no price data found  (1d 2006-01-03 -> 2008-09-15)

1 Failed download:
['LEHMQ']: possibly delisted; no price data found  (1d 2006-01-03 -> 2008-09-15)
$BSC: possibly delisted; no price data found  (1d 2006-01-03 -> 2008-03-17)

1 Failed download:
['BSC']: possibly delisted; no price data found  (1d 2006-01-03 -> 2008-03-17)


 -> Warning: Yahoo returned empty feed for LEHMQ. Will need manual backup source.
Fetching historical data for BSC (Bear Stearns) from 2006-01-03 to 2008-03-17...
 -> Warning: Yahoo returned empty feed for BSC. Will need manual backup source.
Fetching historical data for SIVB (Silicon Valley Bank) from 2018-01-02 to 2023-03-10...


$SIVB: possibly delisted; no timezone found

1 Failed download:
['SIVB']: possibly delisted; no timezone found


 -> Warning: Yahoo returned empty feed for SIVB. Will need manual backup source.
Fetching historical data for SBNY (Signature Bank) from 2018-01-02 to 2023-03-12...


...

 -> Warning: Yahoo returned empty feed for SBNY. Will need manual backup source.
Fetching historical data for PACW (PacWest Bancorp) from 2018-01-02 to 2023-12-29...


$PACW: possibly delisted; no timezone found

1 Failed download:
['PACW']: possibly delisted; no timezone found


 -> Warning: Yahoo returned empty feed for PACW. Will need manual backup source.
Fetching historical data for CHK (Chesapeake Energy) from 2018-01-02 to 2020-06-30...


$CHK: possibly delisted; no timezone found

1 Failed download:
['CHK']: possibly delisted; no timezone found


 -> Warning: Yahoo returned empty feed for CHK. Will need manual backup source.

Completed fetching patches. Total patches ready: 0 / 6


In [10]:
import os

import numpy as np
import pandas as pd

# Define metadata for the 6 historical crisis assets
patch_metadata = {
    "LEHMQ": {
        "Name": "Lehman Brothers",
        "Category": "Distressed_2008",
        "Sector": "Financials",
    },
    "BSC": {
        "Name": "Bear Stearns",
        "Category": "Distressed_2008",
        "Sector": "Financials",
    },
    "SIVB": {
        "Name": "Silicon Valley Bank",
        "Category": "Distressed_2023",
        "Sector": "Financials",
    },
    "SBNY": {
        "Name": "Signature Bank",
        "Category": "Distressed_2023",
        "Sector": "Financials",
    },
    "PACW": {
        "Name": "PacWest Bancorp",
        "Category": "Distressed_2023",
        "Sector": "Financials",
    },
    "CHK": {
        "Name": "Chesapeake Energy",
        "Category": "Distressed_2020",
        "Sector": "Energy",
    },
}

patch_dir = "../data/patches/"
parquet_path = "../data/raw/market_data_raw.parquet"

# Correct way to sort a DataFrame by date
master_df = master_df.sort_values(by="Date").reset_index(drop=True)

# Correct way to sort a specific datetime Series
sorted_dates = master_df["Date"].sort_values()

# Get the full calendar date range of your main dataset to align index properly
all_dates = master_df["Date"].unique()
all_dates = np.sort(all_dates)

processed_patches = []

for ticker, meta in patch_metadata.items():
    file_path = os.path.join(patch_dir, f"{ticker}.csv")

    if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
        print(f"Processing manual patch for {ticker}...")
        df_p = pd.read_csv(file_path)

        # Standardize columns
        df_p["Date"] = pd.to_datetime(df_p["Date"])
        df_p = df_p.sort_values("Date").drop_duplicates(subset=["Date"])

        df_p["Ticker"] = ticker
        df_p["Name"] = meta["Name"]
        df_p["Category"] = meta["Category"]
        df_p["Sector"] = meta["Sector"]

        # Compute log returns up to bankruptcy
        df_p["Log_Return"] = np.log(df_p["Adj_Close"] / df_p["Adj_Close"].shift(1))

        # Keep only necessary columns
        keep_cols = [
            "Date",
            "Ticker",
            "Name",
            "Category",
            "Sector",
            "Open",
            "High",
            "Low",
            "Close",
            "Adj_Close",
            "Volume",
            "Log_Return",
        ]
        df_p = df_p[[c for c in keep_cols if c in df_p.columns]]

        processed_patches.append(df_p)
        print(
            f" -> Successfully integrated {ticker} ({len(df_p)} trading rows up to collapse)."
        )
    else:
        print(f" -> Notice: {ticker}.csv is missing or empty in {patch_dir}.")

if processed_patches:
    patch_df = pd.concat(processed_patches, ignore_index=True)

    # Drop old partial or empty rows for these specific tickers from master
    patched_symbols = list(patch_metadata.keys())
    master_df = master_df[~master_df["Ticker"].isin(patched_symbols)]

    # Append newly patched data
    master_df = pd.concat([master_df, patch_df], ignore_index=True)
    master_df.sort_values(by=["Ticker", "Date"], inplace=True)

    # Save back to master raw parquet
    master_df.to_parquet(parquet_path, engine="pyarrow", index=False)
    print(
        f"\n[SUCCESS] Master raw parquet updated! Total unique tickers now: {master_df['Ticker'].nunique()}"
    )
else:
    print("\nNo valid patch CSV files found to merge.")

 -> Notice: LEHMQ.csv is missing or empty in ../data/patches/.
 -> Notice: BSC.csv is missing or empty in ../data/patches/.
 -> Notice: SIVB.csv is missing or empty in ../data/patches/.
 -> Notice: SBNY.csv is missing or empty in ../data/patches/.
 -> Notice: PACW.csv is missing or empty in ../data/patches/.
 -> Notice: CHK.csv is missing or empty in ../data/patches/.

No valid patch CSV files found to merge.
